# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [632]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [633]:
df = pd.read_csv(r"data\AviationData.csv", encoding='latin-1', low_memory=False)
print("Dataset loaded succesfully")

print(f"The shape is :{df.shape}")

Dataset loaded succesfully
The shape is :(88889, 31)


In [634]:
df.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


In [635]:
print(df.dtypes)

Event.Id                      str
Investigation.Type            str
Accident.Number               str
Event.Date                    str
Location                      str
Country                       str
Latitude                      str
Longitude                     str
Airport.Code                  str
Airport.Name                  str
Injury.Severity               str
Aircraft.damage               str
Aircraft.Category             str
Registration.Number           str
Make                          str
Model                         str
Amateur.Built                 str
Number.of.Engines         float64
Engine.Type                   str
FAR.Description               str
Schedule                      str
Purpose.of.flight             str
Air.carrier                   str
Total.Fatal.Injuries      float64
Total.Serious.Injuries    float64
Total.Minor.Injuries      float64
Total.Uninjured           float64
Weather.Condition             str
Broad.phase.of.flight         str
Report.Status 

In [636]:
print(df.columns.tolist())

['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date', 'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name', 'Injury.Severity', 'Aircraft.damage', 'Aircraft.Category', 'Registration.Number', 'Make', 'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description', 'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status', 'Publication.Date']


In [637]:
print(f"The total missing values are: {df.isnull().sum().sum()}\n")
print(f"The missing values are: \n{df.isnull().sum()}")

The total missing values are: 565032

The missing values are: 
Event.Id                      0
Investigation.Type            0
Accident.Number               0
Event.Date                    0
Location                     52
Country                     226
Latitude                  54507
Longitude                 54516
Airport.Code              38757
Airport.Name              36185
Injury.Severity            1000
Aircraft.damage            3194
Aircraft.Category         56602
Registration.Number        1382
Make                         63
Model                        92
Amateur.Built               102
Number.of.Engines          6084
Engine.Type                7096
FAR.Description           56866
Schedule                  76307
Purpose.of.flight          6192
Air.carrier               72241
Total.Fatal.Injuries      11401
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Uninjured            5912
Weather.Condition          4492
Broad.phase.of.flight     27165
Report.St

In [638]:
print(f"The total duplicates are: {df.duplicated().sum().sum()}\n")
print(f"The duplicates are: \n{df.duplicated().sum()}")

The total duplicates are: 0

The duplicates are: 
0


In [639]:
print(df.shape)
print(df.head())

(88889, 31)
         Event.Id Investigation.Type Accident.Number  Event.Date  \
0  20001218X45444           Accident      SEA87LA080  1948-10-24   
1  20001218X45447           Accident      LAX94LA336  1962-07-19   
2  20061025X01555           Accident      NYC07LA005  1974-08-30   
3  20001218X45448           Accident      LAX96LA321  1977-06-19   
4  20041105X01764           Accident      CHI79FA064  1979-08-02   

          Location        Country   Latitude   Longitude Airport.Code  \
0  MOOSE CREEK, ID  United States        NaN         NaN          NaN   
1   BRIDGEPORT, CA  United States        NaN         NaN          NaN   
2    Saltville, VA  United States  36.922223  -81.878056          NaN   
3       EUREKA, CA  United States        NaN         NaN          NaN   
4       Canton, OH  United States        NaN         NaN          NaN   

  Airport.Name  ... Purpose.of.flight Air.carrier Total.Fatal.Injuries  \
0          NaN  ...          Personal         NaN                 

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [640]:
print(df["Aircraft.Category"].value_counts(dropna=False))


Aircraft.Category
NaN                  56602
Airplane             27617
Helicopter            3440
Glider                 508
Balloon                231
Gyrocraft              173
Weight-Shift           161
Powered Parachute       91
Ultralight              30
Unknown                 14
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
Name: count, dtype: int64


In [641]:
print(df["Amateur.Built"].value_counts(dropna=False))


Amateur.Built
No     80312
Yes     8475
NaN      102
Name: count, dtype: int64


In [642]:
print(df["Event.Date"].describe())

count          88889
unique         14782
top       1982-05-16
freq              25
Name: Event.Date, dtype: object


In [643]:
#Keep airplanes only
df = df[df["Aircraft.Category"] == "Airplane"]


In [644]:

#Keep proffesionaly built airplanes
df = df[df["Amateur.Built"]=="No"]


In [645]:


#Keep events from 1983 - 2023
df['Event.Date'] = pd.to_datetime(df['Event.Date'],errors="coerce")

df =df[df["Event.Date"].dt.year >= 1983]


In [646]:

print(f"Shape after filtering by aircraft category, events and date are :{df.shape}")

Shape after filtering by aircraft category, events and date are :(21447, 31)


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [647]:
# Inspect missing values
injury_cols = [
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured"
]

print(df[injury_cols].isnull().sum())
print(df["Aircraft.damage"].value_counts(dropna=False))

Total.Fatal.Injuries      2750
Total.Serious.Injuries    2828
Total.Minor.Injuries      2544
Total.Uninjured            711
dtype: int64
Aircraft.damage
Substantial    16990
Destroyed       2316
NaN             1227
Minor            817
Unknown           97
Name: count, dtype: int64


In [648]:
# Replace missing injury counts with 0
for col in injury_cols:
    df[col] = df[col].fillna(0)

Missing values in the injury columns were replaced with 0. With the assumption that when injury counts are not reported, they do not contribute to the injury totals. This also allows the creation of aggregate injury metrics without losing records for further analysis.

In [649]:
#Estimation of total people aboard
df["Total.People"] = (
    df["Total.Fatal.Injuries"] +
    df["Total.Serious.Injuries"] +
    df["Total.Minor.Injuries"] +
    df["Total.Uninjured"]
)

In [650]:
#Create the metric for fatal/serious injuries
df["Fatal.Serious"] = (
    df["Total.Fatal.Injuries"] +
    df["Total.Serious.Injuries"]
)

In [651]:
#Estimate the likelihood or rate of fatal/serious injury occurence

df["Fatal.Serious.Rate"] = (
    df["Fatal.Serious"] /
    df["Total.People"]
)

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [652]:
#Inpect the values for aircraft damage
print(df["Aircraft.damage"].value_counts(dropna=False))

Aircraft.damage
Substantial    16990
Destroyed       2316
NaN             1227
Minor            817
Unknown           97
Name: count, dtype: int64


In [653]:
#Replace missing values with "Unknown"
df["Aircraft.damage"] = df["Aircraft.damage"].fillna("Unknown")

In [654]:
#Create a destroyed indicator
df["Aircraft.Destroyed"] = (
    df["Aircraft.damage"] == "Destroyed")


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)


1. Remove leading and trailing whitespace.
2. Convert all manufacturer names to uppercase to ensure consistent capitalization.
3. Drop missing values since they are just less hence not affecting the dataset.
4. Remove makes with fewer than 50 occurrences for meaningful analysis.

In [655]:
print(df['Make'].head())

4149        Lockheed
4150          Boeing
4171           Piper
4285    De Havilland
5957         Douglas
Name: Make, dtype: str


In [656]:
print(f"The missing rows are: {df["Make"].isnull().sum()}")


The missing rows are: 3


In [657]:
print(f"The total number of makes is: {df["Make"].nunique()}")
print(df["Make"].value_counts().head(20))

The total number of makes is: 1332
Make
CESSNA                4867
PIPER                 2803
Cessna                2279
Piper                 1186
BOEING                1037
BEECH                 1018
Beech                  413
MOONEY                 238
Boeing                 227
CIRRUS DESIGN CORP     218
AIR TRACTOR INC        217
AIRBUS                 215
BELLANCA               158
AERONCA                149
MAULE                  144
Mooney                 125
EMBRAER                123
Air Tractor            117
LUSCOMBE                95
STINSON                 91
Name: count, dtype: int64


In [658]:
#Standardizing the manufacturer names
df["Make"] = (df["Make"].dropna().str.strip().str.upper())

In [659]:
#Occeurences count for each make
make_counts = df["Make"].value_counts()

#Keep makes with 50 appearances and more appearances
valid_makes = make_counts[make_counts >= 50].index

df = df[df["Make"].isin(valid_makes)]


In [660]:
#Confirmation for the minimum appearances in the Make column
print(df["Make"].value_counts().min())

50


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

1. Drop missing model values.
2. Remove leading and trailing whitespace.
3. Convert model names to uppercase for consistency.
4. Inspect the frequency of each make/model combination.
5. Create a new column combining `Make` and `Model` because model names are not guaranteed to be unique across different manufacturers.

In [661]:
#Inspect the column 
print(f"The total missing values in the Model column is: {df["Model"].isnull().sum()}\n")
print( df["Model"].value_counts().head(20))


The total missing values in the Model column is: 13

Model
172          769
737          403
152          316
182          304
172S         276
PA28         273
172N         249
SR22         240
180          213
A36          181
172M         180
150          179
PA-18-150    175
PA-28-140    169
172P         143
140          117
172R         109
170B         107
PA-28-180    105
PA-28-161    102
Name: count, dtype: int64


In [662]:
#Cleaning of the model column
df["Model"]= (df["Model"].dropna().str.strip().str.upper())

In [663]:
#Check if the same model is unique to each make

Model_make_freq = (df.groupby(["Make" , "Model"]).size().reset_index(name="Frequency").sort_values("Frequency", ascending=False))
print(Model_make_freq.head(20))

                    Make      Model  Frequency
871               CESSNA        172        769
540               BOEING        737        403
861               CESSNA        152        316
934               CESSNA        182        304
907               CESSNA       172S        276
1983               PIPER       PA28        273
902               CESSNA       172N        249
919               CESSNA        180        213
901               CESSNA       172M        180
846               CESSNA        150        179
1829               PIPER  PA-18-150        175
1870               PIPER  PA-28-140        169
341                BEECH        A36        164
1277  CIRRUS DESIGN CORP       SR22        144
903               CESSNA       172P        143
844               CESSNA        140        116
905               CESSNA       172R        109
870               CESSNA       170B        107
1876               PIPER  PA-28-180        105
1875               PIPER  PA-28-161        102


In [664]:
#Create a unique aircraft identifier
df["Aircraft.Type"] = df["Make"] + " " + df["Model"]
print(f"The number of unique aircraft make_model is: {df["Aircraft.Type"].nunique()}\n")
print(df["Aircraft.Type"].head(20))

The number of unique aircraft make_model is: 2153

4150             BOEING 747
4171        PIPER PA-28-140
4285     DE HAVILLAND DHC-6
6760         BOEING 727-200
6806              BEECH C35
7084            CESSNA 180K
7708               BEECH 99
8585        PIPER PA-23-250
8865        PIPER PA-18-150
10247          CESSNA R172E
10605           CESSNA 182A
10688           CESSNA 182Q
11638           CESSNA T303
11898            CESSNA 152
12683            CESSNA 441
13114            CESSNA 208
14357       PIPER PA-23-250
14420         GRUMMAN G164A
14421         GRUMMAN G164A
14711         CESSNA TU206F
Name: Aircraft.Type, dtype: str


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [665]:
# Inspect the columns
predictive_columns= [
    "Engine.Type",
    "Weather.Condition",
    "Number.of.Engines",
    "Purpose.of.flight",
    "Broad.phase.of.flight"
]

for column in predictive_columns:
    print(df[column].value_counts(dropna=False).head(20))

Engine.Type
Reciprocating      12836
NaN                 3225
Turbo Prop           931
Turbo Fan            702
Unknown              105
Turbo Jet             71
Geared Turbofan       12
Turbo Shaft            9
UNK                    1
Name: count, dtype: int64
Weather.Condition
VMC    14300
NaN     2424
IMC      906
Unk      186
UNK       76
Name: count, dtype: int64
Number.of.Engines
1.0    13230
2.0     2473
NaN     2091
4.0       67
3.0       26
0.0        5
Name: count, dtype: int64
Purpose.of.flight
Personal                     9845
NaN                          3053
Instructional                2410
Aerial Application            726
Business                      410
Unknown                       305
Positioning                   270
Skydiving                     157
Aerial Observation            147
Other Work Use                121
Banner Tow                     86
Flight Test                    73
Ferry                          72
Executive/corporate            65
Glider Tow  

In [666]:
# Clean the columns
cleaned_columns= [
    "Engine.Type",
    "Weather.Condition",
    "Purpose.of.flight",
    "Broad.phase.of.flight"
]

for column in cleaned_columns:
    df[column]= (df[column].astype("string").str.strip().str.upper().replace({
        "UKNOWN": pd.NA,
        "UNK": pd.NA,
        "N/A": pd.NA,
        "NONE": pd.NA,
        "":pd.NA
    })

    )

In [667]:
# Confirm the Number.of.Engines column data type
print(f"The data type is : {df['Number.of.Engines'].dtypes}\n")
print(f"The missing values in the Number.of.Engines is : {df['Number.of.Engines'].isnull().sum()}")


The data type is : float64

The missing values in the Number.of.Engines is : 2091


In [668]:
# Convert the Number.of.Engines column to numeric because there are missing values hence numeric handles it gracefully than integers
df["Number.of.Engines"] = pd.to_numeric(df["Number.of.Engines"],  errors = "coerce")

#Engine count cannot be negative
df.loc[df["Number.of.Engines"] < 0, "Number.of.Engines"] = pd.NA

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [669]:
not_null_frequency = df.count().sort_values(ascending= False)
print(not_null_frequency.head(20))

Event.Id                  17892
Investigation.Type        17892
Accident.Number           17892
Event.Date                17892
Aircraft.damage           17892
Amateur.Built             17892
Make                      17892
Aircraft.Category         17892
Total.Serious.Injuries    17892
Total.Minor.Injuries      17892
Total.Uninjured           17892
Total.Fatal.Injuries      17892
Aircraft.Destroyed        17892
Total.People              17892
Fatal.Serious             17892
Country                   17891
Location                  17888
Aircraft.Type             17879
Model                     17879
Registration.Number       17728
dtype: int64


In [670]:
#Keep columns with more than 16000 non-nulls since the highest number of non-null is 17892
df = df.loc[:,df.notna().sum() > 16000]

In [671]:
print(df.info())

<class 'pandas.DataFrame'>
Index: 17892 entries, 4150 to 88886
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Event.Id                17892 non-null  str           
 1   Investigation.Type      17892 non-null  str           
 2   Accident.Number         17892 non-null  str           
 3   Event.Date              17892 non-null  datetime64[us]
 4   Location                17888 non-null  str           
 5   Country                 17891 non-null  str           
 6   Injury.Severity         17173 non-null  str           
 7   Aircraft.damage         17892 non-null  str           
 8   Aircraft.Category       17892 non-null  str           
 9   Registration.Number     17728 non-null  str           
 10  Make                    17892 non-null  str           
 11  Model                   17879 non-null  str           
 12  Amateur.Built           17892 non-null  str           
 13 

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [672]:
df.to_csv("final_cleaned_aviation_flight_data.csv" , index=False)

print("Cleaned dataset successfully saved.")

Cleaned dataset successfully saved.


In [673]:
#verificarion it was saved
import os
print(os.getcwd())
print(os.path.exists("final_cleaned_aviation_flight_data.csv"))

c:\Users\HomePC\dsc-course0-m8-lab
True
